# Notebook 02 — Explain mode

Tests the explanation pipeline: given a concept or question, retrieve relevant
course chunks and ask the model to explain in plain language.

**Key difference from quiz mode:** The prompt asks for a *walkthrough*, not a JSON object.
You'll see how changing the prompt shape changes the output style — same model, different register.

**Experiment ideas:**
- Try the same concept with RAG on vs off — see how much context helps
- Try different temperature values — low (0.3) is factual, high (1.0) is more verbose
- Compare how each model handles a topic that ISN'T in the course material

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

MODEL_CHOICE = "cpu_tiny"      # flan-t5-small — runs on CPU, ~80 MB
# MODEL_CHOICE = "small_hf"   # flan-t5-base — ~250 MB, better quality
# MODEL_CHOICE = "larger_hf"  # TinyLlama-1.1B — needs GPU, good quality
# MODEL_CHOICE = "own_model"  # your trained checkpoint

CHECKPOINT_PATH = "../teaching_assistant/checkpoints/step_005000.pt"
RAG_INDEX_DIR   = "../teaching_assistant/rag_index"

# The topic or question to explain
EXPLAIN_REQUEST = "Can you explain how backpropagation works?"

# How much context to give the model (in characters, roughly 1 token ≈ 4 chars)
CONTEXT_CHARS = 600

# Generation settings
TEMPERATURE  = 0.7   # 0.3 = focused/safe, 1.0 = creative/verbose
MAX_TOKENS   = 300

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL LOADER — compatible with transformers >= 4.40
#
# text2text-generation was removed from the pipeline registry
# in newer transformers. We use the model + tokenizer directly
# instead — it's the same thing under the hood, just explicit.
# ═══════════════════════════════════════════════════════════
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if MODEL_CHOICE == "cpu_tiny":
    # ── flan-t5-small ──────────────────────────────────────
    # T5 is an encoder-decoder (seq2seq) model.
    # We encode the prompt with the encoder, then let the
    # decoder generate the output autoregressively.
    # ~80 MB, runs in seconds on CPU.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
    _mdl.eval()
    # Note: T5 stays on CPU (device=-1 was the old pipeline arg)

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "small_hf":
    # ── flan-t5-base ───────────────────────────────────────
    # Same architecture, 3× more parameters. Better at following
    # structured instructions (e.g. "output only JSON").
    # ~250 MB. CPU works but is slow; GPU recommended.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
    _mdl = _mdl.to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "larger_hf":
    # ── TinyLlama-1.1B-Chat ────────────────────────────────
    # Decoder-only (same family as GPT). Uses a chat template
    # so the prompt is wrapped in <|user|> / <|assistant|> tags.
    # ~2.2 GB VRAM — fits easily on a 3080 laptop.
    # Swap model_id for "microsoft/phi-2" (2.7B) for better quality.
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    _tok = AutoTokenizer.from_pretrained(model_id)
    _mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        # Apply the chat template so TinyLlama knows this is a user message
        messages  = [{"role": "user", "content": prompt}]
        formatted = _tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tok(formatted, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
            )
        # Decode only the newly generated tokens (skip the prompt)
        new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
        return _tok.decode(new_ids, skip_special_tokens=True)

elif MODEL_CHOICE == "own_model":
    # ── Your trained model from train.py ───────────────────
    # Uses tiktoken gpt2 BPE tokenizer (same as training).
    import sys
    sys.path.insert(0, "../teaching_assistant")
    from rag_pipeline import load_model
    import tiktoken
    _own_model, _own_tok, _own_cfg = load_model(CHECKPOINT_PATH, device)
    _enc = tiktoken.get_encoding("gpt2")

    def generate(prompt, max_new_tokens=200):
        ids = _enc.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)
        out = _own_model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=50,
            stop_token=_enc.eot_token,
        )
        new_ids = out[0, len(ids):].tolist()
        return _enc.decode(new_ids).replace("<|endoftext|>", "").strip()

print(f"Model ready: {MODEL_CHOICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD RAG
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "../teaching_assistant")

retriever = None
try:
    from retriever import Retriever
    retriever = Retriever(index_dir=RAG_INDEX_DIR)
    print("RAG loaded")
except Exception as e:
    print(f"RAG unavailable ({e}) — will explain from model memory only")

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPLAIN FUNCTION
# Two prompt styles:
#   with_rag=True  → "Context: {text}\nRequest: {q}\nResponse:"
#   with_rag=False → "Request: {q}\nResponse:"
# Compare both to see how much the course material helps.
# ═══════════════════════════════════════════════════════════

def explain(request, with_rag=True, verbose=False):
    context_str = ""
    chunks_used = []

    if with_rag and retriever is not None:
        # Pull top-3 chunks most relevant to the request
        chunks_used = retriever.query(request, top_k=3)
        context_str = "\n---\n".join(c["text"][:CONTEXT_CHARS//3] for c in chunks_used)

    if context_str:
        prompt = f"Context:\n{context_str}\n\nRequest: {request}\nResponse:"
    else:
        prompt = f"Request: {request}\nResponse:"

    if verbose:
        print("=== PROMPT SENT TO MODEL ===")
        print(prompt[:500] + ("..." if len(prompt) > 500 else ""))
        print("=" * 40)

    answer = generate(prompt)
    return answer, chunks_used


# ── Run it ─────────────────────────────────────────────────
print(f"Request: {EXPLAIN_REQUEST}\n")
answer, chunks = explain(EXPLAIN_REQUEST, with_rag=True, verbose=True)
print("\n=== MODEL RESPONSE ===")
print(answer)

In [ ]:
# ═══════════════════════════════════════════════════════════
# A/B COMPARISON: with RAG vs without RAG
# Run this cell to directly compare the two.
# This is the core experiment for your report.
# ═══════════════════════════════════════════════════════════

request = EXPLAIN_REQUEST   # change this to any topic

print("[A] WITHOUT RAG (model memory only)")
print("-" * 40)
answer_a, _ = explain(request, with_rag=False)
print(answer_a)

print("\n[B] WITH RAG (grounded in lecture notes)")
print("-" * 40)
answer_b, chunks = explain(request, with_rag=True)
print(answer_b)

if chunks:
    print("\n[Sources used by RAG]")
    for i, c in enumerate(chunks, 1):
        print(f"  {i}. score={c['score']:.3f} | {c.get('breadcrumb','?')}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# BATCH: explain multiple topics and inspect results
# ═══════════════════════════════════════════════════════════

topics = [
    "What is dropout?",
    "Explain batch normalization.",
    "How does gradient descent work?",
]

for topic in topics:
    print(f"\n{'═'*55}\nRequest: {topic}\n{'═'*55}")
    ans, _ = explain(topic, with_rag=True)
    print(ans[:500])   # truncate for readability